# **Подготовка функций парсинга**

In [ ]:
# 1. Функция для зарплаты (приведение к рублям)
def get_salary(arg):
    try:
        # Извлекаем число и валюту
        salary = float(re.sub(r'[^0-9.]', '', arg))
        curr = arg.split('.')[-1].strip().lower()
        # Конвертируем по курсу из словаря currency_rate
        rate = currency_rate.get(curr, 1.0)
        return salary * rate
    except:
        return 0

# 2. Функция для опыта работы (в месяцах)
def get_experience(arg):
    try:
        # Регулярным выражением ищем годы и месяцы
        years = re.search(r'(\d+)\s+г', arg)
        months = re.search(r'(\d+)\s+м', arg)
        total = 0
        if years: total += int(years.group(1)) * 12
        if months: total += int(months.group(1))
        return total
    except:
        return 0

# 3. Функция для городов (категории)
def get_city(arg):
    if 'москва' in arg: return [1, 0, 0, 0]
    if 'санкт-петербург' in arg: return [0, 1, 0, 0]
    # Здесь можно добавить логику для миллионников
    return [0, 0, 0, 1]

# **Преобразование всей таблицы в массивы**

In [ ]:
def extract_row_data(row):
    # Извлекаем все параметры и объединяем в один вектор
    gender_age = get_gender_age(row[COL_SEX_AGE]) # Нужно дописать по аналогии
    city = get_city(row[COL_CITY].lower())
    exp = get_experience(row[COL_EXP])
    # ... добавляем остальные параметры (занятость, график и т.д.)
    return np.hstack([gender_age, city, exp]) # Собираем всё в один массив

# Создаем обучающую выборку
x_train_data = np.array([extract_row_data(row) for row in df.values])
y_train = np.array([get_salary(s) for s in df['ЗП']])

# **Создание и обучение нейросети**

In [ ]:
# Вход для числовых данных
input_numeric = Input(shape=(x_train_data.shape[1],))
x1 = Dense(128, activation='relu')(input_numeric)
x1 = BatchNormalization()(x1)

# Вход для текстовых данных (должность)
input_text = Input(shape=(max_len,))
x2 = Embedding(max_words, 50)(input_text)
x2 = LSTM(64)(x2)

# Объединяем ветки
combined = concatenate([x1, x2])
outputs = Dense(1, activation='linear')(combined) # Регрессия: 1 нейрон на выходе

model = Model(inputs=[input_numeric, input_text], outputs=outputs)
model.compile(optimizer='adam', loss='mse', metrics=['mae'])

# Обучение
history = model.fit([x_train_data, x_train_text], y_train,
                    epochs=20, batch_size=32, validation_split=0.2)